# 1. Environment Setup and Hyperparameters

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random
import numpy as np
!pip install snownlp -q
from snownlp import SnowNLP
from collections import Counter
import math

batch_size = 64        # Number of sequences processed in parallel
block_size = 12        # Maximum context length
max_iters = 5000        # Total training iterations
eval_interval = 100      # Evaluate loss every N iterations
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'

eval_iters = 200       # Number of batches used to estimate loss

# Transformer architecture parameters
n_embd = 256           # Embedding dimension
n_head = 8            # Number of attention heads
n_layer = 6           # Number of Transformer blocks
dropout = 0.1          # Dropout rate

SPECIAL_END_TOKEN = "<END>"   # Special token used to separate names

torch.manual_seed(1337)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 23.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# 2. Dataset Loading and Preprocessing

In [ ]:
# Download Chinese names corpus
!wget https://github.com/hanlincl/Chinese-Names-Corpus/raw/refs/heads/master/Chinese_Names_Corpus/Chinese_Names_Corpus%EF%BC%88120W%EF%BC%89.txt

# Read the dataset
with open('Chinese_Names_Corpus（120W）.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Remove empty lines
lines = [line.strip() for line in lines if line.strip()]

# Add end token to each name
lines = [name + SPECIAL_END_TOKEN for name in lines]

# Shuffle dataset
random.seed(1337)
random.shuffle(lines)

--2026-03-09 22:22:27--  https://github.com/hanlincl/Chinese-Names-Corpus/raw/refs/heads/master/Chinese_Names_Corpus/Chinese_Names_Corpus%EF%BC%88120W%EF%BC%89.txt
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/hanlincl/Chinese-Names-Corpus/refs/heads/master/Chinese_Names_Corpus/Chinese_Names_Corpus%EF%BC%88120W%EF%BC%89.txt [following]
--2026-03-09 22:22:28--  https://raw.githubusercontent.com/hanlincl/Chinese-Names-Corpus/refs/heads/master/Chinese_Names_Corpus/Chinese_Names_Corpus%EF%BC%88120W%EF%BC%89.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12049918 (11M) [text/plain]
Saving to: 

# 3. Vocabulary Construction

In [ ]:
# Merge into one long text sequence
text = "".join(lines)

# Extract unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create mappings between characters and indices
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

# Encoder: string → integer sequence
encode = lambda s: [stoi[c] for c in s]

# Decoder: integer sequence → string
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("豪玉"))
print(decode([1905, 1319]))

[1905, 1319]
豪玉


# 4. Train / Validation / Test Split and Batch Sampling


In [ ]:
total_names = len(lines)    # Total number of names
n1_names = int(0.8 * total_names)   # 80% of total names for training
n2_names = int(0.9 * total_names)   # 90% of total names (80%-90% for validation)

# Split name list
train_names = lines[:n1_names]
val_names = lines[n1_names:n2_names]
test_names = lines[n2_names:]

# Merge names in each set into text sequence separately
train_text = "".join(train_names)
val_text = "".join(val_names)
test_text = "".join(test_names)

# Encode each set's text into tensor separately
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)
test_data = torch.tensor(encode(test_text), dtype=torch.long)

# Batch Sampling
def get_batch(split):
    """ Sample a batch of data for training/evaluation """

    # Select which dataset to sample from
    if split == 'train':
        data = train_data
    elif split == 'val':
        data = val_data
    else:
        data = test_data

    # Random starting indices
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Input sequence
    x = torch.stack([data[i:i+block_size] for i in ix])

    # Target sequence (shifted by 1)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    # Move tensors to GPU if available
    x, y = x.to(device), y.to(device)
    return x, y

# 5. Transformer Architecture and Loss Evaluation

In [ ]:
class Head(nn.Module):
    """ Single self-attention head """

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Lower triangular matrix for causal masking
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        # Compute attention scores
        wei = q @ k.transpose(-2,-1) * C**-0.5

        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        v = self.value(x)

        # Weighted sum of values
        out = wei @ v

        return out


class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Apply attention with residual connection
        x = x + self.sa(self.ln1(x))
        # Apply feed-forward with residual connection
        x = x + self.ffwd(self.ln2(x))
        return x


# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # Token embedding: Map each character to a vector
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Position embedding: Encode positional information
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Transformer blocks
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        # Final LayerNorm
        self.ln_f = nn.LayerNorm(n_embd)
        # Output layer: Map back to vocab_size logits
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb      # (B,T,C)
        x = self.blocks(x)         # (B,T,C)
        x = self.ln_f(x)          # (B,T,C)
        logits = self.lm_head(x)      # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, end_token_id=None):
        """ Generate a new sequence of characters based on the current context """
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            if end_token_id is not None and idx_next.item() == end_token_id:
                break

            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

# Loss Evaluation
@torch.no_grad()
def estimate_loss():
    """ Estimate average loss on train and validation sets """

    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()
    return out

# 6. Evaluation Metrics Filtering

In [21]:
# Frequencies Distribution
def get_top_20_frequencies(dataset):
    generated_names = [name for name in dataset.split('<END>') if len(name) >= 2]
    filter_chars = {'<','E','N','D','>',' ','\n','\t','\r'}
    clean_name_list = []
    for name in generated_names:
        clean_name = ''.join([c for c in name if c not in filter_chars])
        if len(clean_name) < 2 or not all('\u4e00' <= c <= '\u9fff' for c in clean_name):
            continue
        clean_name_list.append(clean_name)
    all_given_name_chars = []
    for clean_name in clean_name_list:
        all_given_name_chars.extend(list(clean_name[1:]))
    top20 = Counter(all_given_name_chars).most_common(20)
    print("\nTop 20")
    for i, (char, count) in enumerate(top20, 1):
        print(f"{i:02d}. {char} ({count} times)")
    return top20

def compare_top_20(gen_list, real_list):
    """ Overlap between generated and real high-frequency chars """

    gen_chars = set([item[0] for item in gen_list]) if gen_list else set()
    real_chars = set([item[0] for item in real_list]) if real_list else set()
    common_chars = gen_chars.intersection(real_chars)
    model_only = gen_chars - real_chars
    real_only = real_chars - gen_chars

    overlap_count = len(common_chars)
    overlap_rate = (overlap_count / 20) * 100

    print(f"TOP 20 comparison report")
    print(f"Common characters ({overlap_count} characters):")
    print(f" {' '.join(list(common_chars))}")
    print(f"\nmodel preferred ({len(model_only)} characters):")
    print(f" {' '.join(list(model_only))}")
    print(f"\nmodel missed ({len(real_only)} characters):")
    print(f" {' '.join(list(real_only))}")
    print(f"common characters rate in Top 20: {overlap_rate:.1f}%")


# Sentimental Analysis
def get_name_sentiment_score(name):
    """ Sentiment scoring via SnowNLP """

    # English Translation: Here is a name: {name}
    # Because SnowNLP is trained in chinese text,
    # so the activation context need to be chinese.
    context_text = f"这个名字是：{name}"
    try:
        s = SnowNLP(context_text)
        score_0_to_1 = s.sentiments

        # Map to 0-5 scale
        score_0_to_5 = round(score_0_to_1 * 5, 2)
        return score_0_to_5
    except:
        return 0.0

def filter_positive_names(generated_text, threshold=3):
    """ Filter generated names with high sentiment """

    # Separate by <END> to obtain the original list of names.
    raw_names = generated_text.split(SPECIAL_END_TOKEN)
    # Filter invalid names (empty/length not met)
    valid_names = []
    for name in raw_names:
        name = name.strip()
        if 2 <= len(name) <= 4 and all('\u4e00' <= c <= '\u9fff' for c in name):
            valid_names.append(name)
    # Deduplication
    valid_names = list(set(valid_names))

    # SnowNLP Scoring + Filtering
    positive_names = []
    for name in valid_names:
        score = get_name_sentiment_score(name)
        if score >= threshold:
            positive_names.append((name, score))

    # Sort by score from highest to lowest.
    positive_names.sort(key=lambda x: x[1], reverse=True)
    return positive_names


def calculate_score_proportion(scores):
    bins = [0, 1.25, 2.5, 3.75, 5]
    labels = [
        '(0-1.25)',
        '(1.25-2.5)',
        '(2.5-3.75)',
        '(3.75-5)'
    ]

    counts, _ = np.histogram(scores, bins=bins)
    proportions = (counts / len(scores)) * 100

    for label, pct in zip(labels, proportions):
        print(f"{label}：{pct:.1f}%")

    pos_pct = np.sum(proportions[2:])
    print(f"positive>=2.5: {pos_pct:.1f}%")

# 7. Model Training

In [ ]:
model = BigramLanguageModel()
m = model.to(device)

# Print total number of parameters in millions
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# Define optimizer: AdamW with learning rate from hyperparameters
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training loop: iterate max_iters times
for iter in range(max_iters):

    # Periodically evaluate loss
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of input-output pairs from training data
    xb, yb = get_batch('train')

    # Forward pass: compute logits and loss
    logits, loss = model(xb, yb)

    # Zero gradients before backward pass
    optimizer.zero_grad(set_to_none=True)
    # Backward pass: compute gradients
    loss.backward()
    # Update model parameters using gradients
    optimizer.step()

5.905124 M parameters
step 0: train loss 7.9121, val loss 7.9133
step 100: train loss 2.1811, val loss 2.1698
step 200: train loss 2.1303, val loss 2.1343
step 300: train loss 2.1210, val loss 2.1246
step 400: train loss 2.1218, val loss 2.1165
step 500: train loss 2.1115, val loss 2.1112
step 600: train loss 2.1045, val loss 2.1127
step 700: train loss 2.1008, val loss 2.1019
step 800: train loss 2.0952, val loss 2.0987
step 900: train loss 2.0815, val loss 2.0899
step 1000: train loss 2.0796, val loss 2.0870
step 1100: train loss 2.0768, val loss 2.0791
step 1200: train loss 2.0786, val loss 2.0809
step 1300: train loss 2.0737, val loss 2.0786
step 1400: train loss 2.0667, val loss 2.0743
step 1500: train loss 2.0676, val loss 2.0575
step 1600: train loss 2.0599, val loss 2.0584
step 1700: train loss 2.0635, val loss 2.0657
step 1800: train loss 2.0624, val loss 2.0610
step 1900: train loss 2.0440, val loss 2.0515
step 2000: train loss 2.0516, val loss 2.0509
step 2100: train loss 2.

# 8. Name Generation

In [ ]:
# Initialize context: Create a (1, 1) tensor of zeros as the starting token
context = torch.zeros((1,1), dtype=torch.long, device=device)

# Identify the end token ID from the character-to-index mapping
end_token_id = None
if '>' in stoi:
    end_token_id = stoi['>']
elif '<' in stoi:
    end_token_id = stoi['<']
else:
    print("")

# Generate token sequences using the model
generated_ids = m.generate(
    context,
    max_new_tokens=2000,
    end_token_id=None
)[0].tolist()

# Decode the generated IDs back into human-readable text
generated_text = decode(generated_ids)

# Split the text into raw names based on the special end separator
raw_name_list = generated_text.split(SPECIAL_END_TOKEN)
filter_chars = {'<', 'E', 'N', 'D', '>', ' ', '\n', '\t', '\r'}

valid_names = []
for name in raw_name_list:
    # Remove all noise characters from the current string
    clean_name = ''.join([c for c in name if c not in filter_chars])
    # Validate the name based on length and Chinese character encoding (Unicode range)
    if 2 <= len(clean_name) <= 4 and all('\u4e00' <= c <= '\u9fff' for c in clean_name):
        valid_names.append(clean_name)

# Remove duplicates while preserving the generation order
valid_names = list(dict.fromkeys(valid_names))

# Output the first 100 valid names
top_100_names = valid_names[:100]
print("\nThe first 100 valid names generated:")
for idx, name in enumerate(top_100_names, start=1):
    print(f"{idx}. {name}")


The first 100 valid names generated:
1. 陈艺林
2. 林亚琛
3. 吴万祥
4. 邱渡
5. 李慧侠
6. 许燕
7. 蔡利鑫
8. 梁晓雯
9. 郭志利
10. 邓从颖
11. 扈璐璐
12. 李兴勋
13. 许日喜
14. 张若渊
15. 龚久
16. 王祖韬
17. 韩伟伟
18. 罗晨阳
19. 周良彬
20. 耿艳玲
21. 俞晓霞
22. 王乐秀
23. 杜天雄
24. 王桂琴
25. 徐淑英
26. 林西林
27. 李春能
28. 张传堂
29. 郭宏志
30. 石依依
31. 澄国玉
32. 叶正雷
33. 陈玉芝
34. 戚妮
35. 姚有林
36. 李含月
37. 徐晓航
38. 芦瑞玉
39. 施少娟
40. 马英伟
41. 李能清
42. 梁永法
43. 何建寿
44. 刘东佳
45. 王兴鹰
46. 杨晓民
47. 夏纹
48. 夏春源
49. 吕天心
50. 何炫
51. 曾玉仙
52. 宗文娟
53. 霍金芳
54. 李立基
55. 孙明琴
56. 胡军伟
57. 毛树珍
58. 戴剑锋
59. 方嘉澍
60. 本丽红
61. 潘思铭
62. 蔡朝敏
63. 王尤文
64. 李沛敏
65. 苗晨辉
66. 万玉莲
67. 钟海莲
68. 刘淳敏
69. 彭仁豪
70. 周弘语
71. 黄松亭
72. 王绰
73. 程文玉
74. 任胜军
75. 熊顺华
76. 马步人
77. 徐万天
78. 韩家玲
79. 强雅娜
80. 汪继中
81. 莫汶文
82. 董俊川
83. 游恬
84. 张学润
85. 杨林玲
86. 张懿
87. 曹兴明
88. 朱正秀
89. 熊梅娟
90. 康爱玉
91. 廖秀庚
92. 董潜
93. 谢为民
94. 李兴贺
95. 唐金鑫
96. 黄海泓
97. 邢子铭
98. 李才萍
99. 杨耀福
100. 吴亚山


# 9. Evaluation Metrics

## 9.1 Frequencies Distribution

In [ ]:
# Decode test data to text
test_text = decode(test_data.tolist())

# Get Top 20 of the generation and test data
real_top20 = get_top_20_frequencies(test_text)    # Function from module 6
gen_top20 = get_top_20_frequencies(generated_text)

# Compare overlap between generated and real top 20 characters
compare_top_20(gen_top20, real_top20)


Top 20
01. 文 (3388 times)
02. 华 (2962 times)
03. 明 (2883 times)
04. 晓 (2202 times)
05. 玉 (2086 times)
06. 国 (1991 times)
07. 伟 (1983 times)
08. 海 (1884 times)
09. 志 (1860 times)
10. 平 (1815 times)
11. 丽 (1753 times)
12. 春 (1749 times)
13. 红 (1735 times)
14. 建 (1721 times)
15. 林 (1657 times)
16. 金 (1621 times)
17. 云 (1621 times)
18. 军 (1594 times)
19. 英 (1587 times)
20. 永 (1493 times)

Top 20
01. 玉 (12 times)
02. 文 (11 times)
03. 晓 (9 times)
04. 海 (7 times)
05. 伟 (6 times)
06. 艳 (6 times)
07. 林 (5 times)
08. 兴 (5 times)
09. 芳 (5 times)
10. 华 (5 times)
11. 玲 (4 times)
12. 秀 (4 times)
13. 天 (4 times)
14. 英 (4 times)
15. 娟 (4 times)
16. 清 (4 times)
17. 金 (4 times)
18. 丽 (4 times)
19. 亮 (4 times)
20. 冰 (4 times)
TOP 20 comparison report
Common characters (10 characters):
 华 英 海 玉 金 丽 林 文 晓 伟

model preferred (10 characters):
 天 兴 清 冰 玲 娟 艳 芳 秀 亮

model missed (10 characters):
 春 平 红 建 国 志 云 军 永 明
common characters rate in Top 20: 50.0%


## 9.2 Perplexity Test

In [ ]:
model.eval()     # Set model to evaluation mode

# Initialize tensor to store batch losses
with torch.no_grad():

    # take 200 batches to get average loss
    test_losses = torch.zeros(200)

    # Loop through batches
    for k in range(200):
        X_test, Y_test = get_batch('test')
        _, loss = model(X_test, Y_test)
        test_losses[k] = loss.item()

    # Average test loss across all batches
    avg_test_loss = test_losses.mean()

    # Perplexity is exp(loss)
    test_ppl = math.exp(avg_test_loss)
    print(f"Final Test Loss: {avg_test_loss:.4f} | Test Perplexity: {test_ppl:.2f}")

Final Test Loss: 2.0142 | Test Perplexity: 7.49


## 9.3 Sentimental analysis

In [22]:
# Count the proportions of each interval for dataset.
all_names = [name.replace(SPECIAL_END_TOKEN, '') for name in lines]
random.seed(1337)
sampled_names = random.sample(all_names, 10000)

# Get sentiment scores using SnowNLP
scores = [get_name_sentiment_score(name) for name in sampled_names]

# Calculate and print proportion of names in each sentiment bin
calculate_score_proportion(scores)

# For generated names
positive_names = filter_positive_names(generated_text)
valid_names = list(set([n.strip() for n in generated_text.split(SPECIAL_END_TOKEN)
                if 2<=len(n.strip())<=4 and all('\u4e00'<=c<='\u9fff' for c in n.strip())]))

# Compute sentiment distribution for valid generated names
if valid_names:
    print("\n")
    calculate_score_proportion([get_name_sentiment_score(n) for n in valid_names])

(0-1.25)：4.4%
(1.25-2.5)：21.3%
(2.5-3.75)：36.3%
(3.75-5)：38.0%
positive>=2.5: 74.3%


(0-1.25)：6.3%
(1.25-2.5)：23.6%
(2.5-3.75)：28.7%
(3.75-5)：41.3%
positive>=2.5: 70.1%
